<h1 align='center'>Movie Recomendation</h1>

In [1]:
!pip install matplotlib plotly --upgrade --quiet
!pip install seaborn --upgrade --quiet
!pip install scikit-learn --upgrade --quiet


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!pip install opendatasets --upgrade --quiet

In [2]:
import numpy as np
import pandas as pd
import opendatasets as op
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [3]:
op.download("https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset


100%|██████████| 228M/228M [00:01<00:00, 168MB/s]  


In [4]:
os.listdir("./the-movies-dataset")

['movies_metadata.csv',
 'credits.csv',
 'keywords.csv',
 'links.csv',
 'ratings_small.csv',
 'links_small.csv',
 'ratings.csv']

In [5]:
movies_metadata=pd.read_csv("./the-movies-dataset/movies_metadata.csv")
keywords=pd.read_csv("./the-movies-dataset/keywords.csv")

/tmp/ipykernel_1609/310089810.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata=pd.read_csv("./the-movies-dataset/movies_metadata.csv")


In [7]:
movies_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [ ]:
cols_to_drop = [
    'budget', 'revenue', 'homepage', 'production_companies', 
    'spoken_languages', 'status', 'video', 'adult', 
    'belongs_to_collection', 'imdb_id', 'original_language', 
    'original_title', 'popularity', 'production_countries', 'release_date','runtime'
]

movies_metadata = movies_metadata.drop(columns=cols_to_drop, errors='ignore').copy()

In [12]:
movies_metadata

,genres,id,overview,poster_path,tagline,title,vote_average,vote_count
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...",/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,NaN,Toy Story,7.7,5415.0
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,/e64sOI48hQXyru7naBFyssKFxVd.jpg,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0
...,...,...,...,...,...,...,...,...
45461,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",439050,Rising and falling between a man and woman.,/jldsYflnId4tTWPx8es3uzsB1I8.jpg,Rising and falling between a man and woman,Subdue,4.0,1.0
45462,"[{'id': 18, 'name': 'Drama'}]",111109,An artist struggles to finish his work while a...,/xZkmxsNmYXJbKVsTRLLx3pqGHx7.jpg,NaN,Century of Birthing,9.0,3.0
45463,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",67758,"When one of her hits goes wrong, a professiona...",/d5bX92nDsISNhu3ZT69uHwmfCGw.jpg,A deadly game of wits.,Betrayal,3.8,6.0
45464,[],227506,"In a small town live two brothers, one a minis...",/aorBPO7ak8e8iJKT5OcqYxU3jlK.jpg,NaN,Satan Triumphant,0.0,0.0


In [15]:
movies_metadata['tagline'] = movies_metadata['tagline'].fillna('')
movies_metadata['overview'] = movies_metadata['overview'].fillna('')

In [19]:
movies_metadata=movies_metadata.drop(columns='poster_path')

In [ ]:
import ast

def parse_genres(genre_str):
    try:
        genres_list = ast.literal_eval(genre_str)
        if isinstance(genres_list, list):
            return ' '.join([g['name'] for g in genres_list if 'name' in g])
    except (ValueError, SyntaxError):
        pass
    return ''

movies_metadata['genres_clean'] = movies_metadata['genres'].fillna('[]').apply(parse_genres)

In [25]:
movies_metadata.drop(columns='genres')

,id,overview,tagline,title,vote_average,vote_count,content,genres_clean
0,862,"Led by Woody, Andy's toys live happily in his ...",,Toy Story,7.7,5415.0,"Led by Woody, Andy's toys live happily in his...",Animation Comedy Family
1,8844,When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0,Roll the dice and unleash the excitement! When...,Adventure Fantasy Family
2,15602,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0,Still Yelling. Still Fighting. Still Ready for...,Romance Comedy
3,31357,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0,Friends are the people who let you be yourself...,Comedy Drama Romance
4,11862,Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0,Just When His World Is Back To Normal... He's ...,Comedy
...,...,...,...,...,...,...,...,...
45461,439050,Rising and falling between a man and woman.,Rising and falling between a man and woman,Subdue,4.0,1.0,Rising and falling between a man and woman Ris...,Drama Family
45462,111109,An artist struggles to finish his work while a...,,Century of Birthing,9.0,3.0,An artist struggles to finish his work while ...,Drama
45463,67758,"When one of her hits goes wrong, a professiona...",A deadly game of wits.,Betrayal,3.8,6.0,A deadly game of wits. When one of her hits go...,Action Drama Thriller
45464,227506,"In a small town live two brothers, one a minis...",,Satan Triumphant,0.0,0.0,"In a small town live two brothers, one a mini...",


In [27]:
movies_metadata['content'] = movies_metadata['genres_clean'] + ' ' + movies_metadata['tagline'] + ' ' + movies_metadata['overview']

In [28]:
movies_metadata

,genres,id,overview,tagline,title,vote_average,vote_count,content,genres_clean
0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",862,"Led by Woody, Andy's toys live happily in his ...",,Toy Story,7.7,5415.0,"Animation Comedy Family Led by Woody, Andy's ...",Animation Comedy Family
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",8844,When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0,Adventure Fantasy Family Roll the dice and unl...,Adventure Fantasy Family
2,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",15602,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0,Romance Comedy Still Yelling. Still Fighting. ...,Romance Comedy
3,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",31357,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0,Comedy Drama Romance Friends are the people wh...,Comedy Drama Romance
4,"[{'id': 35, 'name': 'Comedy'}]",11862,Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0,Comedy Just When His World Is Back To Normal.....,Comedy
...,...,...,...,...,...,...,...,...,...
45461,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",439050,Rising and falling between a man and woman.,Rising and falling between a man and woman,Subdue,4.0,1.0,Drama Family Rising and falling between a man ...,Drama Family
45462,"[{'id': 18, 'name': 'Drama'}]",111109,An artist struggles to finish his work while a...,,Century of Birthing,9.0,3.0,Drama An artist struggles to finish his work ...,Drama
45463,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",67758,"When one of her hits goes wrong, a professiona...",A deadly game of wits.,Betrayal,3.8,6.0,Action Drama Thriller A deadly game of wits. W...,Action Drama Thriller
45464,[],227506,"In a small town live two brothers, one a minis...",,Satan Triumphant,0.0,0.0,"In a small town live two brothers, one a min...",


In [30]:
movies_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   genres        45466 non-null  object 
 1   id            45466 non-null  object 
 2   overview      45466 non-null  object 
 3   tagline       45466 non-null  object 
 4   title         45460 non-null  object 
 5   vote_average  45460 non-null  float64
 6   vote_count    45460 non-null  float64
 7   content       45466 non-null  object 
 8   genres_clean  45466 non-null  object 
dtypes: float64(2), object(7)
memory usage: 3.1+ MB


In [31]:
movies_metadata['vote_count'] = pd.to_numeric(movies_metadata['vote_count'], errors='coerce').fillna(0)
movies_metadata = movies_metadata.sort_values('vote_count', ascending=False).head(10000).reset_index(drop=True)

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(movies_metadata['content'])

print("Cleaned DataFrame shape:", movies_metadata.shape)
print("TF-IDF Matrix shape:", tfidf_matrix.shape)

Cleaned DataFrame shape: (10000, 9)
TF-IDF Matrix shape: (10000, 5000)


In [35]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix shape:", cosine_sim.shape)

Cosine Similarity Matrix shape: (10000, 10000)


In [ ]:
def search_movies(query, top_n=10):
    # 1. Convert the user's text prompt into the exact same TF-IDF vector format
    query_vec = tfidf.transform([query])
    
    # 2. Calculate similarity between user query vector and all 10,000 movie vectors
    sim_scores = linear_kernel(query_vec, tfidf_matrix).flatten()
    
    # 3. Sort indices by highest similarity score
    top_indices = sim_scores.argsort()[::-1][:top_n]
    
    # 4. Return top matching movies
    return movies_metadata[['title', 'vote_average', 'genres_clean', 'overview']].iloc[top_indices]

In [54]:
# 1. Create a reverse mapping of titles to DataFrame indices
indices = pd.Series(movies_metadata.index, index=movies_metadata['title']).drop_duplicates()

# 2. Recommendation function
def get_recommendations(title, cosine_sim=cosine_sim):
    if title not in indices:
        return search_movies(title)
    
    # Get the index of the input movie
    idx = indices[title]
    
    # Get pairwise similarity scores for all movies with that movie
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort movies based on similarity scores (highest to lowest)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the scores of the 10 most similar movies (ignore index 0, which is the movie itself)
    sim_scores = sim_scores[1:11]
    
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    
    # Return top 10 most similar movie titles and vote averages
    return movies_metadata[['title', 'vote_average', 'genres_clean']].iloc[movie_indices]

In [56]:
get_recommendations('happy movies')


,title,vote_average,genres_clean,overview
3552,Deep Red,7.6,Horror Mystery Thriller,A musician witnesses the murder of a famous ps...
4390,The Suicide Shop,6.2,Comedy Animation,A city where suicide is the only growing busin...
1686,Ed Wood,7.3,Comedy Drama History,"The mostly true story of the legendary ""worst ..."
1350,Cinema Paradiso,8.2,Drama Romance,"A filmmaker recalls his childhood, when he fel..."
9682,Extreme Movie,3.8,Comedy,A sketch comedy movie about the joys and embar...
9504,Ti ricordi di me?,6.9,Romance Comedy,The Rascal and The Scatterbrain - is their lov...
6421,Michael Jackson's Thriller,8.1,Horror Music,A night at the movies turns into a nightmare w...
4424,Rabbit Hole,6.8,Drama,Life for a happy couple is turned upside down ...
1435,Happy Gilmore,6.5,Comedy,Failed hockey player-turned-golf whiz Happy Gi...
6145,Another Year,7.0,Comedy Drama,Mike Leigh’s much praised 2010 tragicomical dr...


In [60]:
import joblib

# 1. Export cleaned DataFrame
joblib.dump(movies_metadata, 'movie_df.pkl')

# 2. Export fitted TF-IDF Vectorizer
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

# 3. Export TF-IDF Matrix
joblib.dump(tfidf_matrix, 'tfidf_matrix.pkl')

# 4. Export Cosine Similarity Matrix
joblib.dump(cosine_sim, 'cosine_sim.pkl')

print("Exported all files using joblib in .pkl format!")

Exported all files using joblib in .pkl format!
